# 01 — Kasus Campak dan Perlindungan Vaksin Indonesia (D1)

Notebook ini menyiapkan data untuk bagian **Perkembangan Kasus** dan **Perlindungan Vaksin** di dashboard.

**Hasil:**
1. `indonesia_tahun.csv` — kasus dilaporkan dan cakupan MCV1/MCV2 Indonesia, 2016–2025.
2. `kemenkes_2025_2026.csv` — rekap kasus suspek Kemenkes 2025–2026 (diisi manual dari rilis resmi).

**Sumber data:**
- `reported-cases-of-measles.csv` — Our World in Data, dari WHO Global Health Observatory. https://ourworldindata.org/grapher/reported-cases-of-measles
- `who_coverage_mcv1_mcv2.csv` — WHO Immunization Data Portal, estimasi WHO/UNICEF (WUENIC). https://immunizationdata.who.int/global/wiise-detail-page/measles-vaccination-coverage

**Kesepakatan definisi:**
- Kasus WHO/OWID = kasus **dilaporkan**; data Kemenkes = kasus **suspek**. Keduanya disimpan di tabel terpisah dan tidak digabung di satu grafik.
- Cakupan imunisasi hanya memakai kategori **WUENIC** (ADMIN dan OFFICIAL dibuang).
- Persen ditulis sebagai angka 0–100.

In [1]:
from pathlib import Path
import pandas as pd

# Struktur repo: masukan dibaca dari data/raw/, keluaran ditulis ke data/clean/.
# Notebook bisa dijalankan dari folder notebooks/ atau dari root repo.
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

TAHUN_AWAL, TAHUN_AKHIR = 2016, 2025
TARGET = 95

for nama in ["reported-cases-of-measles.csv", "who_coverage_mcv1_mcv2.csv"]:
    print(nama, "->", "ADA" if (RAW / nama).exists() else "TIDAK DITEMUKAN")

reported-cases-of-measles.csv -> ADA
who_coverage_mcv1_mcv2.csv -> ADA


## 1. Kasus campak dilaporkan (OWID/WHO)

File berisi semua negara sejak 1974. Ambil baris Indonesia (`Code = IDN`) untuk 2016 ke atas. Data kasus tahunan baru tersedia sampai 2024.

In [2]:
kasus_raw = pd.read_csv(RAW / "reported-cases-of-measles.csv")
print(kasus_raw.columns.tolist())

kolom_kasus = [c for c in kasus_raw.columns if c not in ("Entity", "Code", "Year")][0]
kasus = (
    kasus_raw[(kasus_raw["Code"] == "IDN") & (kasus_raw["Year"] >= TAHUN_AWAL)]
    .rename(columns={"Code": "iso3", "Year": "tahun", kolom_kasus: "kasus_dilaporkan"})
    [["iso3", "tahun", "kasus_dilaporkan"]]
)
kasus

['Entity', 'Code', 'Year', 'Measles - number of reported cases']


,iso3,tahun,kasus_dilaporkan
4113,IDN,2016,6962
4114,IDN,2017,9035
4115,IDN,2018,5300
4116,IDN,2019,1965
4117,IDN,2020,524
4118,IDN,2021,394
4119,IDN,2022,7704
4120,IDN,2023,18063
4121,IDN,2024,6328


## 2. Cakupan imunisasi campak MCV1 dan MCV2 (WHO/UNICEF)

File WHO berisi semua negara, semua jenis vaksin, dan beberapa kategori cakupan. Filter:
- `CODE = IDN`
- `COVERAGE_CATEGORY = WUENIC` (estimasi resmi WHO/UNICEF)
- `ANTIGEN` = `MCV1` (dosis 1) dan `MCV2` (dosis 2)

In [3]:
cakupan_raw = pd.read_csv(RAW / "who_coverage_mcv1_mcv2.csv", low_memory=False)
cakupan_raw.columns = [c.strip().upper() for c in cakupan_raw.columns]
print(cakupan_raw.columns.tolist())
print("Kategori cakupan:", cakupan_raw["COVERAGE_CATEGORY"].dropna().unique()[:10])

for kol in ["CODE", "ANTIGEN", "COVERAGE_CATEGORY"]:
    cakupan_raw[kol] = cakupan_raw[kol].astype(str).str.strip().str.upper()

cakupan = cakupan_raw[
    (cakupan_raw["CODE"] == "IDN")
    & (cakupan_raw["COVERAGE_CATEGORY"] == "WUENIC")
    & (cakupan_raw["ANTIGEN"].isin(["MCV1", "MCV2"]))
    & (cakupan_raw["YEAR"].between(TAHUN_AWAL, TAHUN_AKHIR))
]

# Pastikan tidak ada baris ganda per tahun dan antigen sebelum pivot
assert not cakupan.duplicated(["YEAR", "ANTIGEN"]).any(), "Ada baris ganda, cek filter"

cakupan = (
    cakupan.pivot(index="YEAR", columns="ANTIGEN", values="COVERAGE")
    .rename(columns={"MCV1": "mcv1_wuenic", "MCV2": "mcv2_wuenic"})
    .rename_axis(index="tahun", columns=None)
    .reset_index()
)
cakupan

['GROUP', 'GROUP_NAME', 'CODE', 'NAME', 'YEAR', 'ANTIGEN', 'ANTIGEN_DESCRIPTION', 'COVERAGE_CATEGORY', 'COVERAGE_CATEGORY_DESCRIPTION', 'TARGET_NUMBER', 'DOSES', 'COVERAGE']
Kategori cakupan: ['ADMIN' 'OFFICIAL' 'WUENIC']


,tahun,mcv1_wuenic,mcv2_wuenic
0,2016,88.0,55.0
1,2017,90.0,63.0
2,2018,89.0,67.0
3,2019,88.0,71.0
4,2020,76.0,60.0
5,2021,72.0,53.0
6,2022,92.0,71.0
7,2023,82.0,69.0
8,2024,85.0,82.0
9,2025,85.0,78.0


## 3. Gabungkan menjadi `indonesia_tahun`

- `selisih_mcv1_mcv2` = MCV1 − MCV2, yaitu porsi anak yang sudah dapat dosis 1 tetapi belum dosis 2 (dalam poin persen).
- `kurang_dari_95_mcv1` dan `kurang_dari_95_mcv2` = `True` jika cakupan di bawah ambang kekebalan kelompok 95%.
- Kasus 2025 kosong karena data tahunan WHO/OWID baru sampai 2024. Angka 2025 ada di tabel Kemenkes (kasus suspek).

In [4]:
tahun = pd.DataFrame({"tahun": range(TAHUN_AWAL, TAHUN_AKHIR + 1)})
indonesia = (
    tahun.merge(kasus[["tahun", "kasus_dilaporkan"]], on="tahun", how="left")
    .merge(cakupan, on="tahun", how="left")
)
indonesia.insert(0, "iso3", "IDN")
indonesia.insert(1, "negara", "Indonesia")

indonesia["kasus_dilaporkan"] = indonesia["kasus_dilaporkan"].astype("Int64")
indonesia["mcv1_wuenic"] = indonesia["mcv1_wuenic"].round(0).astype("Int64")
indonesia["mcv2_wuenic"] = indonesia["mcv2_wuenic"].round(0).astype("Int64")
indonesia["selisih_mcv1_mcv2"] = indonesia["mcv1_wuenic"] - indonesia["mcv2_wuenic"]
indonesia["kurang_dari_95_mcv1"] = indonesia["mcv1_wuenic"] < TARGET
indonesia["kurang_dari_95_mcv2"] = indonesia["mcv2_wuenic"] < TARGET

indonesia = indonesia[[
    "iso3", "negara", "tahun", "kasus_dilaporkan", "mcv1_wuenic", "mcv2_wuenic",
    "selisih_mcv1_mcv2", "kurang_dari_95_mcv1", "kurang_dari_95_mcv2",
]]
indonesia

,iso3,negara,tahun,kasus_dilaporkan,mcv1_wuenic,mcv2_wuenic,selisih_mcv1_mcv2,kurang_dari_95_mcv1,kurang_dari_95_mcv2
0,IDN,Indonesia,2016,6962,88,55,33,True,True
1,IDN,Indonesia,2017,9035,90,63,27,True,True
2,IDN,Indonesia,2018,5300,89,67,22,True,True
3,IDN,Indonesia,2019,1965,88,71,17,True,True
4,IDN,Indonesia,2020,524,76,60,16,True,True
5,IDN,Indonesia,2021,394,72,53,19,True,True
6,IDN,Indonesia,2022,7704,92,71,21,True,True
7,IDN,Indonesia,2023,18063,82,69,13,True,True
8,IDN,Indonesia,2024,6328,85,82,3,True,True
9,IDN,Indonesia,2025,<NA>,85,78,7,True,True


## 4. Cek terhadap angka pembanding

Angka pembanding dari Rincian Tugas: kasus terendah 394 (2021), tertinggi 18.063 (2023), 6.328 (2024); MCV2 2025 = 78.

In [5]:
k = indonesia.set_index("tahun")
assert k.loc[2021, "kasus_dilaporkan"] == 394
assert k.loc[2023, "kasus_dilaporkan"] == 18063
assert k.loc[2024, "kasus_dilaporkan"] == 6328
assert k["kasus_dilaporkan"].min() == 394 and k["kasus_dilaporkan"].idxmin() == 2021
assert k["kasus_dilaporkan"].max() == 18063 and k["kasus_dilaporkan"].idxmax() == 2023
assert k.loc[2025, "mcv2_wuenic"] == 78
print("Semua angka pembanding cocok.")

Semua angka pembanding cocok.


## 5. Rekap kasus suspek Kemenkes 2025–2026 (manual)

Diisi dari rilis resmi Kemenkes, bukan dari file WHO. Data 2026 bersifat **sementara** (sampai minggu epidemiologi ke-7), jadi tidak boleh dibandingkan langsung dengan total 2025.

- `jumlah_klb` = KLB campak **terkonfirmasi laboratorium**. Pada 2026 juga tercatat 21 KLB suspek (lihat kolom `catatan`).

In [6]:
kemenkes = pd.DataFrame([
    {
        "tahun": 2025,
        "periode": "Januari-Desember 2025",
        "suspek": 63769,
        "terkonfirmasi": 11094,
        "kematian": 69,
        "jumlah_klb": 116,
        "catatan": "KLB terkonfirmasi lab di 89 kabupaten/kota, 16 provinsi; suspek naik 147% dari 2024",
        "sumber": "https://www.kemkes.go.id/id/kemenkes-waspadai-dinamika-campak-nasional-dan-global ; jumlah KLB: https://jogja.antaranews.com/berita/808926/kemenkes-menggencarkan-imunisasi-campak-dan-rubella-minggu-depan",
        "tanggal_rilis": "2026-02-26",
    },
    {
        "tahun": 2026,
        "periode": "s.d. minggu ke-7 2026 (sementara)",
        "suspek": 8224,
        "terkonfirmasi": 572,
        "kematian": 4,
        "jumlah_klb": 13,
        "catatan": "13 KLB terkonfirmasi lab dan 21 KLB suspek di 17 kabupaten/kota, 11 provinsi",
        "sumber": "https://www.kemkes.go.id/id/kemenkes-waspadai-dinamika-campak-nasional-dan-global",
        "tanggal_rilis": "2026-02-26",
    },
])
kemenkes

,tahun,periode,suspek,terkonfirmasi,kematian,jumlah_klb,catatan,sumber,tanggal_rilis
0,2025,Januari-Desember 2025,63769,11094,69,116,"KLB terkonfirmasi lab di 89 kabupaten/kota, 16...",https://www.kemkes.go.id/id/kemenkes-waspadai-...,2026-02-26
1,2026,s.d. minggu ke-7 2026 (sementara),8224,572,4,13,13 KLB terkonfirmasi lab dan 21 KLB suspek di ...,https://www.kemkes.go.id/id/kemenkes-waspadai-...,2026-02-26


## 6. Simpan hasil

Kedua CSV ini diunggah ke Google Sheets `Data Dashboard Campak` (tab `indonesia_tahun` dan `kemenkes_2025_2026`).

In [7]:
indonesia.to_csv(CLEAN / "indonesia_tahun.csv", index=False)
kemenkes.to_csv(CLEAN / "kemenkes_2025_2026.csv", index=False)
print("Tersimpan:", CLEAN / "indonesia_tahun.csv", CLEAN / "kemenkes_2025_2026.csv")

Tersimpan: ..\data\clean\indonesia_tahun.csv ..\data\clean\kemenkes_2025_2026.csv


## 7. Ringkasan angka untuk temuan

Angka di bawah dipakai untuk menulis 3 kalimat temuan ② dan ③.

In [8]:
puncak = k["kasus_dilaporkan"].idxmax()
print(f"Kasus terendah: {k['kasus_dilaporkan'].min():,} ({k['kasus_dilaporkan'].idxmin()})")
print(f"Kasus tertinggi: {k['kasus_dilaporkan'].max():,} ({puncak})")
print(f"Kenaikan 2021 -> 2023: {k.loc[2023, 'kasus_dilaporkan'] / k.loc[2021, 'kasus_dilaporkan']:.0f} kali lipat")
print(f"MCV1 tertinggi 2016-2025: {k['mcv1_wuenic'].max()} ({k['mcv1_wuenic'].idxmax()})")
print(f"Tahun dengan MCV1 >= 95: {int((~k['kurang_dari_95_mcv1']).sum())}")
print(f"Tahun dengan MCV2 >= 95: {int((~k['kurang_dari_95_mcv2']).sum())}")
print(f"Terendah saat pandemi: MCV1 {k.loc[2020:2021, 'mcv1_wuenic'].min()}, MCV2 {k.loc[2020:2021, 'mcv2_wuenic'].min()}")
print(f"2025: MCV1 {k.loc[2025, 'mcv1_wuenic']}, MCV2 {k.loc[2025, 'mcv2_wuenic']}, selisih {k.loc[2025, 'selisih_mcv1_mcv2']} poin")

Kasus terendah: 394 (2021)
Kasus tertinggi: 18,063 (2023)
Kenaikan 2021 -> 2023: 46 kali lipat
MCV1 tertinggi 2016-2025: 92 (2022)
Tahun dengan MCV1 >= 95: 0
Tahun dengan MCV2 >= 95: 0
Terendah saat pandemi: MCV1 72, MCV2 53
2025: MCV1 85, MCV2 78, selisih 7 poin
